# Spark SQL Introduction

## Why SQL on Spark?

One of Spark's most powerful features is that it exposes the **exact same distributed computation engine** through two entirely different interfaces:

1. **The DataFrame API** — a programmatic, chainable Python/Scala/Java interface
2. **Spark SQL** — standard ANSI SQL that runs on top of DataFrames

Both interfaces compile down to the **same logical and physical plan** via Spark's Catalyst optimizer. This means a SQL query and its DataFrame equivalent produce byte-for-byte identical execution — there is no performance difference. Choose whichever is clearer for the task at hand.

### The Bridge: Temporary Views

To query a DataFrame with SQL, you first **register it as a temporary view**. Think of a view as a named alias that Spark's SQL engine can look up:

```python
df.createOrReplaceTempView("my_table")
spark.sql("SELECT * FROM my_table")
```

The view lives in the **SparkSession's catalog** and disappears when the session ends. It does *not* copy data — it is merely a pointer to the original DataFrame's RDD partitions.

### What You Will Learn
- Creating and registering temporary views
- Running aggregation queries with SQL
- Writing the equivalent logic with the DataFrame API
- Using CTEs and HAVING clauses
- Inspecting the session catalog

In [ ]:
from pyspark.sql import SparkSession

# Connect to the standalone Spark cluster.
# The master URL points to the Spark master process inside the Docker network.
# appName appears in the Spark UI under the "Running Applications" list.
spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("04 - Spark SQL Introduction")
    .config("spark.sql.shuffle.partitions", "4")  # keep partition count small for local dev
    .getOrCreate()
)

# Print the Spark version so we know exactly which runtime we are on
print(f"Spark version : {spark.version}")
print(f"Python version: {spark.sparkContext.pythonVer}")

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Define an explicit schema — always prefer this over inferSchema for production code
# because inference requires an extra scan of the data.
sales_schema = StructType([
    StructField("order_id",  IntegerType(), nullable=False),
    StructField("product",   StringType(),  nullable=False),
    StructField("category",  StringType(),  nullable=False),
    StructField("quantity",  IntegerType(), nullable=False),
    StructField("price",     DoubleType(),  nullable=False),
])

# Sample sales data covering three categories.
# Real-world datasets would be loaded from Parquet / Delta / a database.
sales_data = [
    (1001, "Laptop",          "Electronics", 2,  899.99),
    (1002, "USB-C Hub",       "Electronics", 5,   29.99),
    (1003, "Mechanical Keyboard", "Electronics", 3, 119.99),
    (1004, "Monitor 27in",    "Electronics", 1,  349.99),
    (1005, "Webcam HD",       "Electronics", 4,   79.99),
    (1006, "T-Shirt",         "Clothing",    10,  19.99),
    (1007, "Denim Jeans",     "Clothing",     6,  59.99),
    (1008, "Winter Jacket",   "Clothing",     2, 149.99),
    (1009, "Running Shoes",   "Clothing",     3,  89.99),
    (1010, "Wool Socks",      "Clothing",    15,   9.99),
    (1011, "Organic Coffee",  "Food",        20,  12.99),
    (1012, "Granola Bars",    "Food",        30,   4.99),
    (1013, "Olive Oil 500ml", "Food",         8,  11.49),
    (1014, "Dark Chocolate",  "Food",        25,   3.99),
    (1015, "Almond Butter",   "Food",        12,   7.49),
]

df = spark.createDataFrame(sales_data, schema=sales_schema)

# Quick sanity check — show the inferred types
df.printSchema()
print(f"Total rows: {df.count()}")

In [ ]:
# Register the DataFrame as a temp view so Spark SQL can reference it by name.
# createOrReplaceTempView is idempotent — safe to re-run during development.
df.createOrReplaceTempView("sales")

# A plain SELECT via spark.sql() returns a NEW DataFrame — nothing is executed yet.
# .show() triggers the actual computation (an Action).
spark.sql("SELECT * FROM sales LIMIT 5").show()

In [ ]:
# Revenue by category: quantity * price gives line-item revenue,
# SUM aggregates across all orders in that category,
# ORDER BY DESC surfaces the highest-revenue category first.
spark.sql("""
    SELECT
        category,
        ROUND(SUM(quantity * price), 2) AS revenue
    FROM sales
    GROUP BY category
    ORDER BY revenue DESC
""").show()

## DataFrame API vs SQL — Two Roads, Same Destination

Every SQL clause has a direct DataFrame counterpart:

| SQL clause | DataFrame method |
|------------|------------------|
| `SELECT col, expr AS alias` | `.select("col", expr.alias("alias"))` |
| `WHERE condition` | `.filter(condition)` or `.where(condition)` |
| `GROUP BY col` | `.groupBy("col")` |
| `SUM(...)` | `.agg(F.sum(...))` |
| `ORDER BY col DESC` | `.orderBy(F.col("col").desc())` |
| `LIMIT n` | `.limit(n)` |

Both approaches go through **Catalyst** — Spark's query optimizer — which rewrites the logical plan into the most efficient physical plan. Run `df.explain(True)` on either result and you will see identical physical plans.

**Rule of thumb:**  
- Use **SQL** when you are dealing with complex joins/CTEs that are cleaner to read as text, or when collaborating with analysts who know SQL but not Python.  
- Use the **DataFrame API** when you are building reusable pipelines, applying conditional logic, or composing transformations programmatically.

In [ ]:
import pyspark.sql.functions as F

# Exact DataFrame equivalent of the SQL query in cell-05.
# The result should be byte-for-byte identical to the SQL version.
(
    df
    .groupBy("category")                                           # GROUP BY category
    .agg(
        F.round(F.sum(F.col("quantity") * F.col("price")), 2)     # SUM(quantity * price)
         .alias("revenue")
    )
    .orderBy(F.col("revenue").desc())                              # ORDER BY revenue DESC
    .show()
)

In [ ]:
# More complex SQL: use a CTE to pre-compute order totals,
# then apply a HAVING clause to filter only high-value orders (> $200),
# and rank products by total spend using a window function.
#
# Key concepts:
#   WITH  cte AS (...)  — Common Table Expression: a named sub-query
#   HAVING               — like WHERE but applied AFTER aggregation
#   RANK() OVER (...)    — window function: rank within a partition

spark.sql("""
    WITH order_totals AS (
        SELECT
            order_id,
            product,
            category,
            ROUND(quantity * price, 2) AS line_total
        FROM sales
    ),
    category_spend AS (
        SELECT
            category,
            product,
            ROUND(SUM(line_total), 2) AS product_revenue
        FROM order_totals
        GROUP BY category, product
        HAVING SUM(line_total) > 50          -- only products with revenue above $50
    )
    SELECT
        category,
        product,
        product_revenue,
        RANK() OVER (PARTITION BY category ORDER BY product_revenue DESC) AS rank_in_category
    FROM category_spend
    ORDER BY category, rank_in_category
""").show(truncate=False)

In [ ]:
# The Catalog is Spark's metadata store — it tracks databases, tables, functions, and columns.
# listTables() returns all temp views and permanent tables in the current database.
print("Tables / views registered in the current SparkSession:")
for table in spark.catalog.listTables():
    print(f"  name={table.name!r:20s}  database={table.database!r:10s}  "
          f"tableType={table.tableType!r:12s}  isTemporary={table.isTemporary}")

# You can also inspect columns of any registered view
print("\nColumns of the 'sales' view:")
for col in spark.catalog.listColumns("sales"):
    print(f"  {col.name:25s}  type={col.dataType:12s}  nullable={col.nullable}")

In [ ]:
# Always stop the SparkSession at the end of the notebook.
# This releases the executor JVM processes on the cluster,
# making resources available for the next application.
spark.stop()
print("SparkSession stopped.")